# Cliff Protection + Multi-Chunk Interaction Experiment

## Pipeline
1. **Cell 1**: Environment Setup (CUDA kernel build)
2. **Cell 2**: Load Model & Data
3. **Cell 3-4**: Cliff perturbation collection (anchor=8, 4) — 이미 완료
5. **Cell 5-6**: Cliff analysis & visualization — 이미 완료
7. **Cell 7**: Multi-chunk actual forward pass — **NEW** (~30min GPU)
   - additive approximation에서 찾은 best/worst rate splits를 실제 inference로 검증
   - interaction effect 존재 여부 확인

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# - CUDA 커널을 현재 GPU arch에 맞게 강제 재빌드
# - 매 세션 시작 시 이 셀부터 실행 (~3min)
# ============================================================
import os, sys, shutil, subprocess, glob

PROJECT_ROOT = "/content/drive/MyDrive/MambaCompression"
MAMBAIC_ROOT = os.path.join(PROJECT_ROOT, "MambaIC")

if not os.path.isdir(PROJECT_ROOT):
    from google.colab import drive
    drive.mount('/content/drive')

assert os.path.isdir(MAMBAIC_ROOT), f"MambaIC not found: {MAMBAIC_ROOT}"
os.chdir(MAMBAIC_ROOT)

# --- GPU arch ---
import torch
prop = torch.cuda.get_device_properties(0)
major, minor = prop.major, prop.minor
print(f"GPU: {prop.name} (sm_{major}{minor})")

# --- 현재 GPU arch 강제 타겟 ---
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"

# --- 기존 모든 흔적 제거 ---
CACHE_DIR = os.path.join(PROJECT_ROOT, "_kernel_cache")
if os.path.isdir(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)

for so in glob.glob("/usr/local/lib/python*/dist-packages/selective_scan_cuda_oflex*"):
    os.remove(so)
    print(f"Removed: {so}")

for mod in list(sys.modules.keys()):
    if 'selective_scan' in mod:
        del sys.modules[mod]

# pip 캐시도 삭제
subprocess.run([sys.executable, "-m", "pip", "cache", "purge"],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# --- pip deps ---
print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "einops", "scipy", "tqdm", "thop", "fvcore", "pybind11"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "seaborn", "compressai", "timm", "pulp"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# --- VMamba CUDA kernel: 완전 클린 빌드 ---
print(f"\nBuilding VMamba kernel for sm_{major}{minor} (clean build)...")
VMAMBA_DIR = os.path.join(PROJECT_ROOT, "VMamba")
if not os.path.isdir(VMAMBA_DIR):
    subprocess.run(["git", "clone", "https://github.com/MzeroMiko/VMamba.git", VMAMBA_DIR])

SELECTIVE_SCAN_DIR = os.path.join(VMAMBA_DIR, "kernels", "selective_scan")
assert os.path.isdir(SELECTIVE_SCAN_DIR), f"Not found: {SELECTIVE_SCAN_DIR}"

# build 디렉토리도 삭제
build_dir = os.path.join(SELECTIVE_SCAN_DIR, "build")
if os.path.isdir(build_dir):
    shutil.rmtree(build_dir)
egg_dirs = glob.glob(os.path.join(SELECTIVE_SCAN_DIR, "*.egg-info"))
for d in egg_dirs:
    shutil.rmtree(d)

result = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     SELECTIVE_SCAN_DIR,
     "--no-build-isolation",
     "--force-reinstall",
     "--no-cache-dir",
     "-q"],
    capture_output=True, text=True,
    env={**os.environ, "TORCH_CUDA_ARCH_LIST": f"{major}.{minor}"})

if result.returncode != 0:
    print("STDOUT:", result.stdout[-300:] if result.stdout else "")
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError("CUDA kernel build failed!")

print("Build OK")

# --- import 검증 ---
import selective_scan_cuda_oflex
print(f"Import OK: selective_scan_cuda_oflex")

# --- 캐시 저장 ---
os.makedirs(CACHE_DIR, exist_ok=True)
with open(os.path.join(CACHE_DIR, "gpu_arch.txt"), "w") as f:
    f.write(f"sm_{major}{minor}")
for so in glob.glob("/usr/local/lib/python*/dist-packages/selective_scan_cuda_oflex*.so"):
    shutil.copy2(so, os.path.join(CACHE_DIR, os.path.basename(so)))
    with open(os.path.join(CACHE_DIR, "paths.txt"), "w") as f:
        f.write(so + "\n")
    print(f"Cached: {so}")

# --- Forward pass 검증 ---
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)
from ModularModels import ModularAE
_net = ModularAE(encoder_type='mamba', decoder_type='transnet',
                 encoded_dim=512, M=32, encoder_layers=2, decoder_layers=2).cuda()
with torch.no_grad():
    _x = torch.randn(1, 2, 32, 32).cuda()
    _y = _net(_x)
    print(f"Forward pass OK: {_x.shape} -> {_y.shape}")
del _net, _x, _y
torch.cuda.empty_cache()

print("\n=== Environment Ready ===")

In [ ]:
# ============================================================
# Cell 2: Load Model & Data
# ============================================================
import importlib, sys, os, argparse
import numpy as np
import pandas as pd
import torch

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

import rpmpq_v2
importlib.reload(rpmpq_v2)
from rpmpq_v2 import (
    build_kernel, _load_model_and_data, get_encoder_block_names,
    RESULTS_CSV,
)

args = argparse.Namespace(
    encoder='mamba', decoder='transnet', encoded_dim=512, M=32,
    encoder_layers=2, decoder_layers=2, use_chunking=False,
    train_path='data/DATA_Htrainout.mat', test_path='data/DATA_Htestout.mat',
    train_key='HT', test_key='HT', no_cuda=False,
    batch_size=256, num_workers=0,
    checkpoint='saved_models/mamba_transnet_L2_dim512_baseline/best.pth',
    aq=8, anchor_bits=16, fc_chunks=32,
)
net, test_loader, norm_params, device = _load_model_and_data(args)
K_d = build_kernel(32, 1.0)
K_a = build_kernel(32, 1.0)

block_names = get_encoder_block_names(net, fc_chunks=32)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Blocks: {len(block_names)}, Samples: {len(test_loader.dataset)}")

# Quick sanity: forward pass 1 batch
with torch.no_grad():
    batch = next(iter(test_loader)).to(device)
    out = net(batch)
    print(f"Forward pass OK: input {batch.shape} -> output {out.shape}")
    del batch, out
    torch.cuda.empty_cache()

print("\nReady.")

In [ ]:
# ============================================================
# Cell 3: Collect Perturbation (anchor=8)
# anchor = all blocks at INT8, perturb each to {16, 4, 2}
# ~2h on Colab GPU
# ============================================================
import importlib, os, sys, time, gc, argparse
import torch

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

# Auto-load model if not in memory (after kernel restart)
if 'net' not in dir():
    print("Model not in memory. Loading...")
    import rpmpq_v2
    importlib.reload(rpmpq_v2)
    from rpmpq_v2 import build_kernel, _load_model_and_data
    args = argparse.Namespace(
        encoder='mamba', decoder='transnet', encoded_dim=512, M=32,
        encoder_layers=2, decoder_layers=2, use_chunking=False,
        train_path='data/DATA_Htrainout.mat', test_path='data/DATA_Htestout.mat',
        train_key='HT', test_key='HT', no_cuda=False,
        batch_size=256, num_workers=0,
        checkpoint='saved_models/mamba_transnet_L2_dim512_baseline/best.pth',
        aq=8, anchor_bits=16, fc_chunks=32,
    )
    net, test_loader, norm_params, device = _load_model_and_data(args)
    K_d = build_kernel(32, 1.0)
    K_a = build_kernel(32, 1.0)
    print(f"Loaded. GPU: {torch.cuda.get_device_name(0)}")

import rpmpq_v2
importlib.reload(rpmpq_v2)
from rpmpq_v2 import collect_per_block_perturbation, RESULTS_CSV

gc.collect(); torch.cuda.empty_cache()

ANCHOR_BITS = 8
pert_csv = os.path.join(RESULTS_CSV, f"cliff_perturbation_anchor{ANCHOR_BITS}.csv")
anc_csv = os.path.join(RESULTS_CSV, f"cliff_anchor{ANCHOR_BITS}.csv")
zeta_csv = os.path.join(RESULTS_CSV, "rpmpq_v2_zeta.csv")

if os.path.exists(pert_csv):
    import pandas as pd
    n = len(pd.read_csv(pert_csv))
    print(f"Already exists: {pert_csv} ({n} rows)")
    print("Delete the file to re-run.")
else:
    t0 = time.time()
    collect_per_block_perturbation(
        model=net, test_loader=test_loader, device=device,
        norm_params=norm_params, snr_list=[10, 20, 30],
        anchor_bits=ANCHOR_BITS,
        bit_options=[16, 8, 4, 2],
        fc_chunks=32, aq_bits=8,
        perturbation_csv=pert_csv,
        anchor_csv=anc_csv,
        zeta_csv=zeta_csv,
        K_d=K_d, K_a=K_a,
    )
    print(f"\nDone in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ============================================================
# Cell 4: Collect Perturbation (anchor=4)
# anchor = all blocks at INT4, perturb each to {16, 8, 2}
# ~2h on Colab GPU
#
# TIP: Cell 3 후 런타임 재시작해도 OK — 모델 자동 로드됨
# ============================================================
import importlib, os, sys, time, gc, argparse
import torch

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

# Auto-load model if not in memory (after kernel restart)
if 'net' not in dir():
    print("Model not in memory. Loading...")
    import rpmpq_v2
    importlib.reload(rpmpq_v2)
    from rpmpq_v2 import build_kernel, _load_model_and_data
    args = argparse.Namespace(
        encoder='mamba', decoder='transnet', encoded_dim=512, M=32,
        encoder_layers=2, decoder_layers=2, use_chunking=False,
        train_path='data/DATA_Htrainout.mat', test_path='data/DATA_Htestout.mat',
        train_key='HT', test_key='HT', no_cuda=False,
        batch_size=256, num_workers=0,
        checkpoint='saved_models/mamba_transnet_L2_dim512_baseline/best.pth',
        aq=8, anchor_bits=16, fc_chunks=32,
    )
    net, test_loader, norm_params, device = _load_model_and_data(args)
    K_d = build_kernel(32, 1.0)
    K_a = build_kernel(32, 1.0)
    print(f"Loaded. GPU: {torch.cuda.get_device_name(0)}")

import rpmpq_v2
importlib.reload(rpmpq_v2)
from rpmpq_v2 import collect_per_block_perturbation, RESULTS_CSV

gc.collect(); torch.cuda.empty_cache()

ANCHOR_BITS = 4
pert_csv = os.path.join(RESULTS_CSV, f"cliff_perturbation_anchor{ANCHOR_BITS}.csv")
anc_csv = os.path.join(RESULTS_CSV, f"cliff_anchor{ANCHOR_BITS}.csv")
zeta_csv = os.path.join(RESULTS_CSV, "rpmpq_v2_zeta.csv")

if os.path.exists(pert_csv):
    import pandas as pd
    n = len(pd.read_csv(pert_csv))
    print(f"Already exists: {pert_csv} ({n} rows)")
    print("Delete the file to re-run.")
else:
    t0 = time.time()
    collect_per_block_perturbation(
        model=net, test_loader=test_loader, device=device,
        norm_params=norm_params, snr_list=[10, 20, 30],
        anchor_bits=ANCHOR_BITS,
        bit_options=[16, 8, 4, 2],
        fc_chunks=32, aq_bits=8,
        perturbation_csv=pert_csv,
        anchor_csv=anc_csv,
        zeta_csv=zeta_csv,
        K_d=K_d, K_a=K_a,
    )
    print(f"\nDone in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ============================================================
# Cell 5: Cliff Protection Analysis (CPU OK)
# GPU 없이 실행 가능. Cell 3, 4 결과만 있으면 됨.
# ============================================================
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
RESULTS_CSV = os.path.join(MAMBAIC_ROOT, "results", "csv")
RESULTS_PLOT = os.path.join(MAMBAIC_ROOT, "results", "plots")
os.makedirs(RESULTS_PLOT, exist_ok=True)

def cosine_sim(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12: return float('nan')
    return float(np.dot(a, b) / (na * nb))


def analyze_cliff_protection(anchor_bits, pert_csv, anc_csv, perf_csv):
    """Analyze which blocks protect against reliability cliff."""
    print(f"\n{'='*70}")
    print(f"  CLIFF PROTECTION: anchor=INT{anchor_bits}")
    print(f"{'='*70}")

    pert = pd.read_csv(pert_csv)
    anc = pd.read_csv(anc_csv)
    perf = pd.read_csv(perf_csv)
    N = len(anc)
    block_names = sorted(pert['block_name'].unique())
    bit_options = sorted(pert['bits'].unique())

    nmse_anc = anc['nmse_linear'].values
    nmse_anc_db = 10 * np.log10(np.mean(nmse_anc) + 1e-15)
    print(f"  Anchor NMSE: {nmse_anc_db:.2f} dB")
    print(f"  Blocks: {len(block_names)}, Bits: {bit_options}, Samples: {N}")

    results_all = {}

    for snr in [10, 20, 30]:
        r_anc = anc[f'rate_{snr}'].values
        r_ref = perf[f'r_perf_{snr}'].values

        print(f"\n--- SNR={snr}dB ---")
        for g in [95, 98, 99]:
            outage_anc = np.mean(r_anc < (g/100) * r_ref)
            print(f"  Anchor outage (g={g/100}): {outage_anc:.4f}")

        for target_bits in bit_options:
            direction = 'UP' if target_bits > anchor_bits else 'DOWN'
            imp_nmse, imp_rate = [], []
            imp_outage = {g: [] for g in [95, 98, 99]}
            valid_blocks = []

            for bname in block_names:
                mask = (pert['block_name'] == bname) & (pert['bits'] == target_bits)
                if mask.sum() == 0: continue
                df_mb = pert[mask].sort_values('sample_idx')
                if len(df_mb) != N: continue

                nmse_pert = df_mb['nmse_linear'].values
                r_pert = df_mb[f'rate_{snr}'].values

                imp_nmse.append(np.mean(nmse_anc - nmse_pert))  # + = improved
                imp_rate.append(np.mean(r_pert - r_anc))        # + = improved
                valid_blocks.append(bname)

                for g in [95, 98, 99]:
                    gamma = g / 100.0
                    out_base = np.mean(r_anc < gamma * r_ref)
                    out_pert = np.mean(r_pert < gamma * r_ref)
                    imp_outage[g].append(out_base - out_pert)   # + = improved

            imp_nmse = np.array(imp_nmse)
            imp_rate = np.array(imp_rate)
            if len(imp_nmse) == 0: continue

            cos_nr = cosine_sim(imp_nmse, imp_rate)
            rho_nr, _ = spearmanr(imp_nmse, imp_rate)

            print(f"\n  [{direction}] {anchor_bits} -> {target_bits}:")
            print(f"    cos(NMSE, Rate)={cos_nr:.4f}  spearman={rho_nr:.4f}")

            for g in [95, 98, 99]:
                imp_out = np.array(imp_outage[g])
                cos_no = cosine_sim(imp_nmse, imp_out)
                cos_ro = cosine_sim(imp_rate, imp_out)
                rho_no, _ = spearmanr(imp_nmse, imp_out)
                print(f"    g={g/100}: cos(NMSE,Out)={cos_no:.4f} "
                      f"cos(Rate,Out)={cos_ro:.4f} "
                      f"spearman(NMSE,Out)={rho_no:.4f}")

            # Top-5 비교
            imp_out_99 = np.array(imp_outage[99])
            top5_out = set(np.argsort(imp_out_99)[-5:])
            top5_nmse = set(np.argsort(imp_nmse)[-5:])
            overlap = len(top5_out & top5_nmse)

            print(f"    Top-5 OUTAGE protectors:")
            for i in np.argsort(imp_out_99)[-5:][::-1]:
                print(f"      {valid_blocks[i]}: out_red={imp_out_99[i]:.5f} "
                      f"nmse_red={imp_nmse[i]:.6f}")
            print(f"    Top-5 NMSE reducers:")
            for i in np.argsort(imp_nmse)[-5:][::-1]:
                print(f"      {valid_blocks[i]}: nmse_red={imp_nmse[i]:.6f} "
                      f"out_red={imp_out_99[i]:.5f}")
            tag = 'SAME' if overlap == 5 else f'DIFFERENT ({5-overlap} differ!)'
            print(f"    Top-5 overlap: {overlap}/5 -> {tag}")

            results_all[(snr, target_bits)] = {
                'blocks': valid_blocks,
                'nmse_red': imp_nmse,
                'rate_gain': imp_rate,
                'outage_red_99': imp_out_99,
            }

    return results_all


# --- Run ---
perf_csv = os.path.join(RESULTS_CSV, "rpmpq_v2_perfect_rates.csv")

for anc_bits, pert_fn, anc_fn in [
    (16, "rpmpq_v2_perturbation.csv", "rpmpq_v2_anchor.csv"),
    (8,  "cliff_perturbation_anchor8.csv", "cliff_anchor8.csv"),
    (4,  "cliff_perturbation_anchor4.csv", "cliff_anchor4.csv"),
]:
    p = os.path.join(RESULTS_CSV, pert_fn)
    a = os.path.join(RESULTS_CSV, anc_fn)
    if os.path.exists(p) and os.path.exists(a):
        analyze_cliff_protection(anc_bits, p, a, perf_csv)
    else:
        print(f"\n[SKIP] anchor={anc_bits}: data not found")

In [ ]:
# ============================================================
# Cell 6: Visualization
# cosine(NMSE, Outage) vs anchor stress → 갈라지면 유효
# ============================================================
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
RESULTS_CSV = os.path.join(MAMBAIC_ROOT, "results", "csv")
RESULTS_PLOT = os.path.join(MAMBAIC_ROOT, "results", "plots")

def cosine_sim(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12: return float('nan')
    return float(np.dot(a, b) / (na * nb))


def get_upgrade_cosines(anchor_bits, pert_csv, anc_csv, perf_csv):
    pert = pd.read_csv(pert_csv)
    anc = pd.read_csv(anc_csv)
    perf = pd.read_csv(perf_csv)
    N = len(anc)
    block_names = sorted(pert['block_name'].unique())
    nmse_anc = anc['nmse_linear'].values

    rows = []
    for snr in [10, 20, 30]:
        r_anc = anc[f'rate_{snr}'].values
        r_ref = perf[f'r_perf_{snr}'].values

        for tb in sorted(pert['bits'].unique()):
            if tb <= anchor_bits: continue  # only upgrades
            imp_n, imp_r, imp_o99 = [], [], []
            for bn in block_names:
                m = (pert['block_name']==bn) & (pert['bits']==tb)
                if m.sum()==0: continue
                df_mb = pert[m].sort_values('sample_idx')
                if len(df_mb)!=N: continue
                imp_n.append(np.mean(nmse_anc - df_mb['nmse_linear'].values))
                imp_r.append(np.mean(df_mb[f'rate_{snr}'].values - r_anc))
                imp_o99.append(
                    np.mean(r_anc < 0.99*r_ref) -
                    np.mean(df_mb[f'rate_{snr}'].values < 0.99*r_ref))

            imp_n, imp_r, imp_o99 = map(np.array, [imp_n, imp_r, imp_o99])
            if len(imp_n) < 3: continue
            rho_no, _ = spearmanr(imp_n, imp_o99)
            rows.append({
                'anchor': anchor_bits, 'target': tb, 'snr': snr,
                'cos_nmse_rate': cosine_sim(imp_n, imp_r),
                'cos_nmse_outage': cosine_sim(imp_n, imp_o99),
                'cos_rate_outage': cosine_sim(imp_r, imp_o99),
                'spearman_nmse_outage': rho_no,
            })
    return pd.DataFrame(rows)


perf_csv = os.path.join(RESULTS_CSV, "rpmpq_v2_perfect_rates.csv")
dfs = []
for ab, pf, af in [
    (16, "rpmpq_v2_perturbation.csv", "rpmpq_v2_anchor.csv"),
    (8, "cliff_perturbation_anchor8.csv", "cliff_anchor8.csv"),
    (4, "cliff_perturbation_anchor4.csv", "cliff_anchor4.csv"),
]:
    pp = os.path.join(RESULTS_CSV, pf)
    ap = os.path.join(RESULTS_CSV, af)
    if os.path.exists(pp) and os.path.exists(ap):
        dfs.append(get_upgrade_cosines(ab, pp, ap, perf_csv))
        print(f"anchor={ab}: OK")
    else:
        print(f"anchor={ab}: MISSING")

if not dfs:
    print("No data found.")
else:
    all_df = pd.concat(dfs, ignore_index=True)
    print("\n" + all_df.to_string(index=False))

    # Plot: upgrade to 16-bit, cosine vs anchor stress
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for idx, snr in enumerate([10, 20, 30]):
        ax = axes[idx]
        sub = all_df[(all_df['snr']==snr) & (all_df['target']==16)].sort_values('anchor', ascending=False)
        if len(sub) == 0: continue

        x = sub['anchor'].values
        ax.plot(x, sub['cos_nmse_outage'], 'o-', label='cos(NMSE, Outage)', color='#d62728', lw=2)
        ax.plot(x, sub['cos_nmse_rate'], 's--', label='cos(NMSE, Rate)', color='#1f77b4', lw=2)
        ax.plot(x, sub['spearman_nmse_outage'], '^:', label='spearman(NMSE, Outage)', color='#2ca02c', lw=2)

        ax.set_xlabel('Anchor (lower = more stressed)', fontsize=12)
        ax.set_ylabel('Correlation', fontsize=12)
        ax.set_title(f'SNR={snr}dB', fontsize=13)
        ax.set_ylim(0.3, 1.05)
        ax.set_xticks(x)
        ax.invert_xaxis()
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0.9, color='gray', ls=':', alpha=0.5, label='_')

    fig.suptitle('Block Ranking: NMSE vs Outage Alignment Under Stress\n(upgrade to 16-bit)', fontsize=14)
    fig.tight_layout()
    path = os.path.join(RESULTS_PLOT, 'cliff_protection_cosine.png')
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\nSaved: {path}")
    print("\ncos < 0.9 = outage ranking diverges from NMSE -> cliff protection works!")

In [ ]:
# ============================================================
# Cell 7: Multi-Chunk Actual Forward Pass (GPU, ~30min)
#
# additive approximation이 아닌 실제 multi-block inference로
# "같은 BOPs에서 다른 FC chunk 조합이 rate를 다르게 주는지" 검증
#
# 방법:
#   - non-FC blocks 고정 (stem=16, mamba=4, proj_conv=4)
#   - FC 32 chunks 중 16개를 2-bit, 16개를 8-bit으로 랜덤 split
#   - 200개 랜덤 split에 대해 실제 forward pass
#   - NMSE vs Rate scatter → spread 있으면 interaction 존재
# ============================================================
import importlib, os, sys, time, gc, argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

# Auto-load model
if 'net' not in dir():
    print("Loading model...")
    import rpmpq_v2
    importlib.reload(rpmpq_v2)
    from rpmpq_v2 import build_kernel, _load_model_and_data, get_encoder_block_names
    args = argparse.Namespace(
        encoder='mamba', decoder='transnet', encoded_dim=512, M=32,
        encoder_layers=2, decoder_layers=2, use_chunking=False,
        train_path='data/DATA_Htrainout.mat', test_path='data/DATA_Htestout.mat',
        train_key='HT', test_key='HT', no_cuda=False,
        batch_size=256, num_workers=0,
        checkpoint='saved_models/mamba_transnet_L2_dim512_baseline/best.pth',
        aq=8, anchor_bits=16, fc_chunks=32,
    )
    net, test_loader, norm_params, device = _load_model_and_data(args)
    K_d = build_kernel(32, 1.0)
    K_a = build_kernel(32, 1.0)
    print(f"Loaded. GPU: {torch.cuda.get_device_name(0)}")

import rpmpq_v2
importlib.reload(rpmpq_v2)
from rpmpq_v2 import get_encoder_block_names, RESULTS_CSV
from train_ae import (apply_precision_policy, restore_fp32_weights,
                      quantize_feedback_torch, calculate_su_miso_rate_mrt)

gc.collect(); torch.cuda.empty_cache()

block_names = get_encoder_block_names(net, fc_chunks=32)
fc_blocks = [b for b in block_names if "fc_part" in b]
non_fc_blocks = [b for b in block_names if "fc_part" not in b]

real_model = net.module if isinstance(net, nn.DataParallel) else net
original_state = {k: v.clone().cpu() for k, v in real_model.state_dict().items()}

min_val, range_val = norm_params
N = len(test_loader.dataset)
snr_list = [10, 20, 30]

print(f"FC chunks: {len(fc_blocks)}, Non-FC: {len(non_fc_blocks)}")
print(f"Samples: {N}")

# --- Helper: evaluate one policy ---
def eval_policy(policy, aq_bits=8):
    real_model.load_state_dict(original_state)
    apply_precision_policy(net, policy, device)

    nmse_all, rates_all = [], {s: [] for s in snr_list}
    with torch.no_grad():
        for batch in test_loader:
            d = batch.to(device)
            z = real_model.encoder(d)
            if aq_bits > 0:
                z = quantize_feedback_torch(z, aq_bits)
            x_hat = real_model.decoder(z)

            h_true = (d * range_val) + min_val - 0.5
            h_hat = (x_hat * range_val) + min_val - 0.5

            err = torch.sum((h_true - h_hat)**2, dim=[1,2,3])
            pwr = torch.sum(h_true**2, dim=[1,2,3])
            nmse_all.extend((err / (pwr + 1e-9)).cpu().numpy().tolist())

            for snr in snr_list:
                r = calculate_su_miso_rate_mrt(h_true, h_hat, snr, device)
                rates_all[snr].extend(r.cpu().numpy().tolist())

    real_model.load_state_dict(original_state)
    return np.array(nmse_all), {s: np.array(v) for s, v in rates_all.items()}

# --- Perfect rates for outage ---
perf_csv = os.path.join(RESULTS_CSV, "rpmpq_v2_perfect_rates.csv")
perf_df = pd.read_csv(perf_csv)
r_ref = {s: perf_df[f"r_perf_{s}"].values for s in snr_list}

# --- Non-FC base policy: stressed state ---
# stem.0=16 (protect), rest non-FC=4 (stressed), FC=varies
NON_FC_POLICY = {}
for bn in non_fc_blocks:
    if bn == "stem.0":
        NON_FC_POLICY[bn] = 16  # always protect stem
    else:
        NON_FC_POLICY[bn] = 4   # stressed

# --- Generate random FC splits + specific splits ---
np.random.seed(42)
N_RANDOM = 200
FC_HIGH = 8
FC_LOW = 2

splits = []

# Random splits: 16 chunks @ FC_HIGH, 16 chunks @ FC_LOW
for i in range(N_RANDOM):
    perm = np.random.permutation(len(fc_blocks))
    high_set = set(perm[:16])
    fc_policy = {}
    for j, bn in enumerate(fc_blocks):
        fc_policy[bn] = FC_HIGH if j in high_set else FC_LOW
    splits.append(("random", {**NON_FC_POLICY, **fc_policy}))

# All FC at high
fc_all_high = {bn: FC_HIGH for bn in fc_blocks}
splits.append(("all_fc_high", {**NON_FC_POLICY, **fc_all_high}))

# All FC at low
fc_all_low = {bn: FC_LOW for bn in fc_blocks}
splits.append(("all_fc_low", {**NON_FC_POLICY, **fc_all_low}))

# Direction-aware split: protect chunks with highest direction ratio
# (from our analysis: fc_part31, fc_part13, fc_part8 etc. are most rate-damaging)
rate_damaging = ["fc_part31", "fc_part13", "fc_part8", "fc_part27", "fc_part3",
                 "fc_part2", "fc_part24", "fc_part12", "fc_part25", "fc_part23",
                 "fc_part5", "fc_part29", "fc_part10", "fc_part30", "fc_part17",
                 "fc_part11"]  # top 16 rate-damaging
fc_dir_aware = {}
for bn in fc_blocks:
    fc_dir_aware[bn] = FC_HIGH if bn in rate_damaging else FC_LOW
splits.append(("direction_aware", {**NON_FC_POLICY, **fc_dir_aware}))

# NMSE-aware split: protect chunks with highest NMSE importance
# (from perturbation data: fc_part6, fc_part18, fc_part20, fc_part9 etc.)
nmse_damaging = ["fc_part6", "fc_part18", "fc_part20", "fc_part9", "fc_part31",
                 "fc_part4", "fc_part16", "fc_part26", "fc_part2", "fc_part0",
                 "fc_part8", "fc_part12", "fc_part13", "fc_part30", "fc_part19",
                 "fc_part29"]  # top 16 NMSE-damaging
fc_nmse_aware = {}
for bn in fc_blocks:
    fc_nmse_aware[bn] = FC_HIGH if bn in nmse_damaging else FC_LOW
splits.append(("nmse_aware", {**NON_FC_POLICY, **fc_nmse_aware}))

print(f"\nTotal policies to evaluate: {len(splits)}")
print(f"  {N_RANDOM} random + 4 special (all_high, all_low, direction_aware, nmse_aware)")
print(f"  Non-FC: stem.0=16, rest=4")
print(f"  FC: 16 chunks @ INT{FC_HIGH}, 16 chunks @ INT{FC_LOW}")
print(f"\nStarting evaluation...")

# --- Evaluate all splits ---
results = []
t0 = time.time()

for idx, (name, policy) in enumerate(splits):
    nmse_arr, rates = eval_policy(policy, aq_bits=8)
    nmse_db = 10 * np.log10(np.mean(nmse_arr) + 1e-15)

    row = {"name": name, "idx": idx, "nmse_db": nmse_db}
    for snr in snr_list:
        row[f"rate_{snr}"] = float(np.mean(rates[snr]))
        row[f"outage99_{snr}"] = float(np.mean(rates[snr] < 0.99 * r_ref[snr]))
        row[f"outage95_{snr}"] = float(np.mean(rates[snr] < 0.95 * r_ref[snr]))

        # cos^2 theta
        cos2 = (2**rates[snr] - 1) / (2**r_ref[snr] - 1 + 1e-12)
        row[f"cos2_{snr}"] = float(np.mean(np.clip(cos2, 0, 1)))

    results.append(row)

    if (idx + 1) % 50 == 0 or name != "random":
        elapsed = time.time() - t0
        print(f"  [{idx+1}/{len(splits)}] {name:20s} NMSE={nmse_db:.2f}dB "
              f"Rate20={row['rate_20']:.4f} cos2_20={row['cos2_20']:.4f} "
              f"({elapsed:.0f}s)")

    gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(results)
csv_path = os.path.join(RESULTS_CSV, "multi_chunk_interaction.csv")
df.to_csv(csv_path, index=False)
print(f"\nSaved: {csv_path}")
print(f"Total time: {(time.time()-t0)/60:.1f} min")

# --- Analysis ---
print("\n" + "=" * 70)
print("  RESULTS: Multi-Chunk Interaction")
print("=" * 70)

rand_df = df[df["name"] == "random"]
special = df[df["name"] != "random"]

for snr in [20]:
    print(f"\n--- SNR={snr}dB ---")
    print(f"  Random splits (n={len(rand_df)}):")
    print(f"    NMSE:    [{rand_df['nmse_db'].min():.3f}, {rand_df['nmse_db'].max():.3f}] dB "
          f"(spread: {rand_df['nmse_db'].max()-rand_df['nmse_db'].min():.3f} dB)")
    print(f"    Rate:    [{rand_df[f'rate_{snr}'].min():.4f}, {rand_df[f'rate_{snr}'].max():.4f}] "
          f"(spread: {rand_df[f'rate_{snr}'].max()-rand_df[f'rate_{snr}'].min():.4f})")
    print(f"    cos2:    [{rand_df[f'cos2_{snr}'].min():.6f}, {rand_df[f'cos2_{snr}'].max():.6f}]")
    print(f"    Out99:   [{rand_df[f'outage99_{snr}'].min():.4f}, {rand_df[f'outage99_{snr}'].max():.4f}]")

    rho, _ = spearmanr(rand_df["nmse_db"], rand_df[f"rate_{snr}"])
    print(f"    Spearman(NMSE, Rate): {rho:.4f}")

    print(f"\n  Special policies:")
    for _, r in special.iterrows():
        print(f"    {r['name']:20s} NMSE={r['nmse_db']:.3f}dB  "
              f"Rate={r[f'rate_{snr}']:.4f}  cos2={r[f'cos2_{snr}']:.6f}  "
              f"Out99={r[f'outage99_{snr}']:.4f}")

    # Within-NMSE-bin rate spread (actual, not additive approx)
    rand_sorted = rand_df.sort_values("nmse_db")
    rand_sorted["nmse_bin"] = pd.qcut(rand_sorted["nmse_db"], 10, labels=False, duplicates="drop")
    print(f"\n  Within-NMSE-bin rate spread (ACTUAL inference):")
    for bid in sorted(rand_sorted["nmse_bin"].unique()):
        sub = rand_sorted[rand_sorted["nmse_bin"] == bid]
        if len(sub) < 5: continue
        spread = sub[f"rate_{snr}"].max() - sub[f"rate_{snr}"].min()
        cos_spread = sub[f"cos2_{snr}"].max() - sub[f"cos2_{snr}"].min()
        print(f"    NMSE~{sub['nmse_db'].mean():.2f}dB (n={len(sub)}): "
              f"rate_spread={spread:.4f}  cos2_spread={cos_spread:.6f}")

# --- Plot ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, snr in enumerate([10, 20, 30]):
    ax = axes[idx]
    ax.scatter(rand_df["nmse_db"], rand_df[f"rate_{snr}"],
               alpha=0.3, s=15, c="gray", label="Random splits")

    for _, r in special.iterrows():
        marker = {"all_fc_high": "^", "all_fc_low": "v",
                  "direction_aware": "*", "nmse_aware": "s"}.get(r["name"], "o")
        color = {"all_fc_high": "green", "all_fc_low": "red",
                 "direction_aware": "blue", "nmse_aware": "orange"}.get(r["name"], "black")
        ax.scatter(r["nmse_db"], r[f"rate_{snr}"],
                   marker=marker, s=150, c=color, edgecolors="black",
                   label=r["name"], zorder=5)

    ax.set_xlabel("NMSE (dB)")
    ax.set_ylabel("Rate (bps/Hz)")
    ax.set_title(f"SNR={snr}dB")
    ax.legend(fontsize=7, loc="lower left")
    ax.grid(True, alpha=0.3)

fig.suptitle("Multi-Chunk FC Split: NMSE vs Rate (actual inference)", fontsize=14)
fig.tight_layout()
path = os.path.join(RESULTS_CSV, "..", "plots", "multi_chunk_interaction.png")
fig.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print(f"\nPlot saved: {path}")
print("\nKey: if direction_aware (blue *) is above the random cloud at same NMSE → interaction works!")